# Recursive eraser!

This application recusively deletes generated pages below the given root

Think: `rm -rf pageroot`

## Load configuration
Be aware not to commit your credentials!

In [ ]:
import yaml
import copy
import logging
log = logging.getLogger(__name__)

with open('private.yaml') as f:
    config = yaml.safe_load(f)

conf_conf = config['confluence']
assert conf_conf
assert len(conf_conf['apiurl']) > 0
space_key = conf_conf['space']
root_page = conf_conf['rootpage']

conf_confidential = copy.deepcopy(config)
conf_confidential['confluence']['password'] = '***'
conf_confidential

In [ ]:
# enable overwrite by test infrastructure
confluence_username = config['confluence']['username']
confluence_password = config['confluence']['password']

The [Confluence API](https://github.com/atlassian-api/atlassian-python-api) is embedded as a **git submodule** in the 'lib' folder next to this notebook.

Use `git submodule update --init` to fetch all submodules after a checkout without `--recursive` option.

If the next cell fails, install confluence-api submodule from the repository root with:
`git submodule add -f https://github.com/atlassian-api/atlassian-python-api.git notebooks/contentfactory/lib/atlassian-python-api`

In [ ]:
import sys
import os
from tqdm.notebook import tqdm_notebook

library = 'lib/atlassian-python-api'
sys.path.insert(0, os.path.abspath(library))
from atlassian import Confluence

In [ ]:
confluence = Confluence(url=config['confluence']['apiurl'], username=confluence_username, password=confluence_password, cloud=True)
root_page_id = confluence.get_page_id(space_key, config['confluence']['rootpage'])
root_page_id

In [ ]:
children = confluence.get_page_child_by_type(root_page_id, limit=100000)
print('Do you really want do delete {count} elements: {head}'.format(count=len(children), 
                                                                     head=str(list(map(lambda x: x['title'], children)))[:250]))

In [ ]:
children[0]

In [ ]:
with tqdm_notebook(total=len(children), dynamic_ncols=True, unit='Page') as pbar:    
    while True:
        children = confluence.get_page_child_by_type(root_page_id, limit=250)
        if len(children) == 0:
            break
        pbar.total = len(children)
        for child in children:
            pbar.set_description('Removing page {}'.format(child['title']))
            confluence.remove_page(child['id'], recursive=True)
            pbar.update(1)

https://developer.atlassian.com/server/confluence/advanced-searching-using-cql/

In [ ]:
#im_pages = confluence.get_all_pages_by_label('im', limit=100000)

from requests.exceptions import HTTPError
url = 'rest/api/content/search'
params = {}
#params['cql'] = 'title ~ "ENTI*" and type={type} AND label="{label}" AND label="im" AND label="entity" AND label="generated"'.format(type="page", label='IM', space=space_key)
params['cql'] = 'label="manual"'.format(type="page", label='IM', space=space_key)
params['cqlcontext'] = '{{ "spaceKey": "{space_key}" }}'.format(space_key=space_key)
params['limit'] = 5
params['max'] = 5
params['expand']='version'
try:
    im_result = confluence.get(url, params=params)
    im_pages = im_result.get("results")
except HTTPError as e:
    log.exception('Failed ' + e.response.content.decode('utf-8'), e)
len(im_pages)

In [ ]:
im_result['_links']

In [ ]:
im_pages[0]

In [ ]:
with tqdm_notebook(total=len(im_pages), desc='Deleting pages ................', dynamic_ncols=True, unit='Page') as pbar:   
    for page in im_pages:
        if space_key in page['_expandable'].get('space'):
            pbar.set_description('Deleting page {}'.format(page['title']))
            try:
                confluence.remove_page(page['id'], recursive=False)
            except Exception as e:
                print('Unable to delete page {}'.format(page['title']))
        else:
            print('Skipping page {} due to misfit of {} in {}'.format(page, space_key, page['_expandable'].get('space')))
        pbar.update(1)

In [ ]:
with tqdm_notebook(total=len(im_pages), ncols=700, unit='Page') as pbar:   
    for page in im_pages:
        if space_key in page['_expandable'].get('space'):
            pbar.set_description('Deleting page {}'.format(page['title']))
            confluence.remove_page(page['id'], recursive=False)
        else:
            print('Skipping page {} due to misfit of {} in {}'.format(page, space_key, page['_expandable'].get('space')))
        pbar.update(1)

In [ ]:
#im_pages = confluence.get_all_pages_by_label('im', limit=100000)

from requests.exceptions import HTTPError
url = 'rest/api/content/search'
params = {}
params['cql'] = '(title ~ "* en - Dokumentation" or title ~ "* Tabellen-Dokumentation en") and type={type} AND label="dm" AND (label="documentation" or label="toc")'.format(type="page", label='IM', space=space_key)
params['cqlcontext'] = '{{ "spaceKey": "{space_key}" }}'.format(space_key=space_key)
params['limit'] = 10000
try:
    im_pages = confluence.get(url, params=params).get("results")
except HTTPError as e:
    log.exception('Failed ' + e.response.content.decode('utf-8'), e)
len(im_pages)

In [ ]:
with tqdm_notebook(total=len(im_pages), dynamic_ncols=True, unit='Page') as pbar:   
    for page in im_pages:
        if space_key in page['_expandable'].get('space'):
            pbar.set_description('Deleting page {}'.format(page['title']))
            confluence.remove_page(page['id'], recursive=False)
        else:
            print('Skipping page {} due to misfit of {} in {}'.format(page, space_key, page['_expandable'].get('space')))
        pbar.update(1)